In [16]:
import coiled

import fsspec
import s3fs
import numpy as np
import rioxarray
import xarray as xr
import fsspec
import pandas as pd
import logging
from flox.xarray import xarray_reduce
import numpy as np
import pytz
import dask
import rasterio
import re
import requests
import sparse
import time
import zarr
from io import BytesIO
from datetime import datetime
from dask.distributed import Client, LocalCluster
from dask.distributed import print
from flox import ReindexArrayType, ReindexStrategy
import hvplot.pandas

import pygwalker as pyg

# Downloaded flox 0.10.3 from https://pypi.org/project/flox/#files usin the source distribution (tar.gz, https://files.pythonhosted.org/packages/0b/b6/5e3d79ef8e3dd3bb1ba656167e2059c0aa000244321995c508db32c7a578/flox-0.10.3.tar.gz)
# Installed in my working conda environment using pip: pip install /home/dagibbs22/flox-0.10.3.tar.gz

# coiled notebook start --region=us-east-1

In [47]:
cluster = coiled.Cluster(
    name="tcl_dask_new_zarrs",
    region="us-east-1", # close to dataset, avoid egress charges
    n_workers=10,
    tags={"project": "LULUCF_zonal_stats"},
    scheduler_vm_types="r7g.xlarge", 
    worker_vm_types="r7g.2xlarge",
    # scheduler_vm_types="x2gd.xlarge", 
    # worker_vm_types="x2gd.xlarge",
    compute_purchase_option="spot_with_fallback"
)

client = cluster.get_client()

Output()

╭────────────────────────── Not Synced with Cluster ───────────────────────────╮
│                 ╷                                                ╷           │
│   Package       │ Error                                          │ Level     │
│ ╶───────────────┼────────────────────────────────────────────────┼─────────╴ │
│   pygwalker     │ Pip check had the following issues that need   │ Warning   │
│                 │ resolving:                                     │           │
│                 │ pygwalker 0.3.17 has requirement               │           │
│                 │ duckdb==0.9.2, but you have duckdb 1.3.0.      │           │
│                 │ pygwalker 0.3.17 has requirement               │           │
│                 │ segment-analytics-python==2.2.3, but you have  │           │
│                 │ segment-analytics-python 2.3.3.                │           │
│   pydantic_core │ pydantic-core~=2.33.2 has no install candidate │ Warning   │
│                 │ for Python 3.12 linux-aarch64 on conda-forge   │           │
│   dtale         │ Pip check had the following issues that need   │ Warning   │
│                 │ resolving:                                     │           │
│                 │ dtale 3.17.0 has requirement dash<=2.18.2;     │           │
│                 │ python_version > "3.7", but you have dash      │           │
│                 │ 3.0.4.                                         │           │
│                 │ dtale 3.17.0 has requirement                   │           │
│                 │ dash-bootstrap-components<=1.7.1;              │           │
│                 │ python_version > "3.0", but you have           │           │
│                 │ dash-bootstrap-components 2.0.3.               │           │
│                 │ dtale 3.17.0 has requirement dash_daq<=0.5.0,  │           │
│                 │ but you have dash-daq 0.6.0.                   │           │
│   awscrt        │ awscrt~=0.26.1 has no install candidate for    │ Warning   │
│                 │ Python 3.12 linux-aarch64 on conda-forge       │           │
│                 ╵                                                ╵           │
╰──────────────────────────────────────────────────────────────────────────────╯

Output()

In [ ]:
client.restart() 

In [ ]:
local_cluster = LocalCluster()  
local_client = Client(local_cluster)
local_client

In [ ]:
local_client.shutdown()

In [4]:
# Conversion of carbon to CO2
C_to_CO2 = 44/12

def timestr():

    # Define the Eastern Time timezone
    eastern = pytz.timezone('US/Eastern')

    # Get the current time in UTC and convert to Eastern Time
    eastern_time = datetime.now(eastern)

    # Format the time as a string
    return eastern_time.strftime("%Y%m%d_%H_%M_%S")

In [5]:
def create_state_node_df(state_node_lookup_table_local, state_node_lookup_table_s3, sheet_name):

    try:
        # Try fetching the file from the S3 URL
        # print(f"Attempting to download file from URL: {spreadsheet}")
        response = requests.get(state_node_lookup_table_s3, timeout=10)
        response.raise_for_status()
        state_node_df = pd.read_excel(BytesIO(response.content), sheet_name=sheet_name)

    except (requests.exceptions.RequestException, Exception) as e:
        print(f"Failed to download file from S3. Falling back to local file. Error: {e}")

        print(f"Reading file from local path: {state_node_lookup_table_local}")
        state_node_df = pd.read_excel(state_node_lookup_table_local, sheet_name=sheet_name)

    return state_node_df

In [6]:
# per https://chatgpt.com/g/g-vK4oPfjfp-coding-assistant/c/682201ec-1f84-800a-a9f9-c9564f613208
def list_folder_uris(base_uri):

    # Initializes S3 filesystem
    fs = s3fs.S3FileSystem(anon=False)  # Set anon=True if public bucket
    
    # Lists all files in the directory
    all_files = fs.ls(base_uri)
    
    # Filters for GeoTIFFs
    tif_files = [f"s3://{f}" for f in all_files if f.endswith(".tif")]
    
    # Converts to a Pandas Series
    series = pd.Series(tif_files)
    
    return series

In [7]:
# Node codes output from model. Covers entire decision tree. Make sure that node codes are right-padded with 0s to 7 digits! 
# Otherwise, only the node codes that are seven digits without 0s will be matched with the node code rasters and output. 
# TODO: I may have accidentally missed some node codes when copying them from the decision tree. Check!
node_codes = np.array([
    1110000, 1120000, 1210000, 1220000, 2111000, 2112000,
    2121100, 2121200, 2122100, 2122200, 2123100, 2123200,
    2124100, 2124200, 2125100, 2125200, 2211100, 2211200, 2212110, 2212120, 
    2212210, 2212220, 2213110, 2213120, 2213210, 2213220,
    2214100, 2214200, 2215100, 2215200, 2221100, 2221200, 
    2222100, 2222200, 2223100, 2223200, 3110000, 3120000, 
    3211111, 3211112, 3211121, 3211122, 3211211, 3211212,
    3211221, 3211222, 3212111, 3212112, 3212121, 3212122,
    3212211, 3212212, 3212221, 3212222, 3221110, 3221120,
    3221210, 3221220, 3222111, 3222112, 3222121, 3222122,
    3222210, 3222220, 4100000, 4210000, 4220000, 4310000,
    4320000, 5100000, 5210000, 5220000, 5310000, 5320000],
dtype=np.uint32)

# GADM v4.1 adm0 IDs (from Solomon Negusse's notebook)
gadm_adm0_ids = np.array([  0.,   4.,   8.,  10.,  12.,  16.,  20.,  24.,  28.,  31.,  32.,
        36.,  40.,  44.,  48.,  50.,  51.,  52.,  56.,  60.,  64.,  68.,
        70.,  72.,  74.,  76.,  84.,  86.,  90.,  92.,  96., 100., 104.,
       108., 112., 116., 120., 124., 132., 136., 140., 144., 148., 152.,
       156., 158., 162., 166., 170., 174., 175., 178., 180., 184., 188.,
       191., 192., 196., 203., 204., 208., 212., 214., 218., 222., 226.,
       231., 232., 233., 234., 238., 239., 242., 246., 248., 250., 254.,
       258., 260., 262., 266., 268., 270., 275., 276., 288., 292., 296.,
       300., 304., 308., 312., 316., 320., 324., 328., 332., 334., 336.,
       340., 348., 352., 356., 360., 364., 368., 372., 376., 380., 384.,
       388., 392., 398., 400., 404., 408., 410., 414., 417., 418., 422.,
       426., 428., 430., 434., 438., 440., 442., 450., 454., 458., 462.,
       466., 470., 474., 478., 480., 484., 492., 496., 498., 499., 500.,
       504., 508., 512., 516., 520., 524., 528., 531., 533., 534., 535.,
       540., 548., 554., 558., 562., 566.,70., 574., 578., 580., 581.,
       583., 584., 585., 586., 591., 598., 600., 604., 608., 612., 616.,
       620., 624., 626., 630., 634., 638., 642., 643., 646., 652., 654.,
       659., 660., 662., 663., 666., 670., 674., 678., 682., 686., 688.,
       690., 694., 702., 703., 704., 705., 706., 710., 716., 724., 728.,
       729., 732., 740., 744., 748., 752., 756., 760., 762., 764., 768.,
       772., 776., 780., 784., 788., 792., 795., 796., 798., 800., 804.,
       807., 818., 826., 831., 832., 833., 834., 840., 850., 854., 858.,
       860., 862., 876., 882., 887., 894.], dtype=np.uint16)

primary_forest_IFL_codes = np.array([0, 1], dtype=np.uint8)

In [8]:
# Extracts file pattern from uri
def parse_pattern_from_uri(uri_series):

    uri = uri_series.values.tolist()[0]
    # print("Parsing URI:", uri)

    # regex per https://chatgpt.com/g/g-vK4oPfjfp-coding-assistant/c/681a538d-55e4-800a-818b-bcf850757ba0
    pattern = r"__([a-zA-Z0-9_]+(?:__?[a-zA-Z0-9_]+)*)_pixel_yr_\d{4}_\d{4}\.tif$"
    match = re.search(pattern, uri)

    if match:
        return match.group(1)
    else:
        return None

In [9]:
def convert_to_coord_dict(flux_results):

    print(f"   Postprocessing {interval}: {timestr()}")
    sparse_data = flux_results.data
    
    dim_names = flux_results.dims
    indices = sparse_data.coords  # tuple of arrays with indices into each dim
    values = sparse_data.data     # non-zero values
    
    coord_dict = {
        dim: flux_results.coords[dim].values[indices[i]]
        for i, dim in enumerate(dim_names)
    }
    coord_dict["value"] = values

    return coord_dict

In [10]:
# Converts flox output to dataframe and does some processing of it
def create_interval_df(coord_dict, state_node_df):

    df = pd.DataFrame(coord_dict)
    # print(df)

    # Replaces numeric values for outputs with names
    df['flux_type'] = df['flux_type'].replace({0: gross_emis_CO2_output_pattern, 1: gross_emis_all_gases_output_pattern, 
                                               2: gross_remv_all_pools_output_pattern, 3: net_flux_output_pattern, 4: "area__ha"})
    # print("with flux_type:", df)
    
    # Classifies the node_codes by larger groupings
    df['node_grp'] = df['state_nodes'].apply(classify_node)
    # print("with classified nodes:", df)

    # Makes node_codes into strings
    df['state_node'] = 'n' + df['state_nodes'].astype(str)
    # print("with n prefix:", df)

    # Adds the interval end year to the dataframe
    df['interval_end'] = interval_end_year
    # print("with interval end year:", df)

    df = df.merge(state_node_df[['state_node_with_prefix', 'meaning']],
              left_on='state_node', right_on='state_node_with_prefix',
              how='left')

    # Converts area from m^2 to ha
    df.loc[df['flux_type'].eq('area__ha'), 'value'] = df['value'] / 10000

    # Drop the helper column if you don't want it
    df.drop(columns=['state_node_with_prefix'], inplace=True)
    
    # print(df)

    return df

In [11]:
# Calculates flux densities (Mg CO2 or CO2e/ha)
# Per https://chatgpt.com/g/g-vK4oPfjfp-coding-assistant/c/682a8c76-0618-800a-a201-18fd404a281f
def calculate_interval_flux_densities(df):

    # Step 1: Filters out area and flux data
    area_df = df[df['flux_type'] == 'area__ha'].copy()
    flux_df = df[df['flux_type'] != 'area__ha'].copy()
    
    # Step 2: Merges flux data with area data on matching keys
    merged = pd.merge(
        flux_df,
        area_df[['state_nodes', 'gadm_adm0', 'interval_end', 'value']],
        on=['state_nodes', 'gadm_adm0', 'interval_end'],
        how='left',
        suffixes=('', '_area')
    )
    # print("merged:" merged)
    
    # Step 3: Computes per-hectare flux (converts CO2 to C)
    merged['value_per_ha'] = merged['value'] / merged['value_area'] / C_to_CO2
    
    # Step 4: Prepares flux density rows to append
    new_rows = merged.copy()
    new_rows['flux_type'] = new_rows['flux_type'] + '__C_per_ha'
    new_rows['value'] = new_rows['value_per_ha']
    new_rows = new_rows.drop(columns=['value_area', 'value_per_ha'])
    # print("new rows:", new_rows)
    
    # Step 5: Appends flux density rows to original dataframe
    result_df = pd.concat([df, new_rows], ignore_index=True)

    return result_df

In [12]:
# Reclassifies state nodes to broad categories
def classify_node(state_node):
    
    node_str = str(state_node)
    first_digit = int(node_str[0])
    # print(first_digit)

    # For broad classes that can be categorized using just the first digit
    one_digit_map = {
        1: 'forest_gain',
        2: 'forest_loss',
        4: 'cropland',
        5: 'grassland'
    }

    # For broad classes that need to be categorized using the first three digits
    three_digit_map = {
        311: 'forest_loss',
        312: 'forest_loss',
        321: 'disturbed_forest',
        322: 'stable_forest'
        # Add more as needed
    }
    
    if first_digit == 3:
        prefix = int(node_str[:3])
        # print(prefix)
        # print(two_digit_map.get(prefix, 'unknown_3x'))
        return three_digit_map.get(prefix, 'unknown_3x')
    else:
        return one_digit_map.get(first_digit, 'unknown')

Code to run zonal stats

In [48]:
# uri components

model_version = "version_0_3_3"
run_date = "20250511"
chunk_size = 10000
EXPECTED_SHAPE = (40000, 40000)  # height (y), width (x)

output_path = f"s3://gfw2-data/climate/AFOLU_flux_model/LULUCF/outputs/{model_version}/"
interval_end_years = [2016]
# interval_end_years = [2020]
# interval_end_years = [2016, 2017, 2018]
# interval_end_years = [2019, 2020, 2021, 2022, 2023]
# interval_end_years = [2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023]

# s3 folders for inputs
gross_emis_CO2_folder = f"{output_path}gross_emissions__all_C_pools__CO2_only__MgCO2/standard_model/annual_intervals/INTERVAL/_pixel_yr/40000_pixels/{run_date}/"
gross_emis_all_gases_folder = f"{output_path}gross_emissions__all_C_pools__all_gases__MgCO2e/standard_model/annual_intervals/INTERVAL/_pixel_yr/40000_pixels/{run_date}/"
gross_remv_all_pools_folder = f"{output_path}gross_removals__all_C_pools__MgCO2/standard_model/annual_intervals/INTERVAL/_pixel_yr/40000_pixels/{run_date}/"
net_flux_all_pools_CO2_folder = f"{output_path}net_flux__all_C_pools__CO2_only__MgCO2/standard_model/annual_intervals/INTERVAL/_pixel_yr/40000_pixels/{run_date}/"
node_folder = f"{output_path}land_state_node/standard_model/annual_intervals/INTERVAL/40000_pixels/{run_date}/"

# Folder where the model output zarrs are stored. They are in their own special outputs folder
zarr_s3_path = f"s3://gfw2-data/climate/AFOLU_flux_model/LULUCF/outputs/{model_version}/zarr/{run_date}/"

# zarrs for layers not from the flux model (only need to created once)
adm0_folder = "s3://gfw2-data/gadm_administrative_boundaries/v4.1/v4.1.64__from_gfw-data-lake/raster/epsg-4326/10/40000/adm0/gdal-geotiff/" #GADM v4.1
adm0_zarr_name = "s3://gfw2-data/climate/AFOLU_flux_model/LULUCF/outputs/contextual_layer_global_zarr/20250604/global_GADM41_adm0_20250604.zarr"

pixel_area_folder = "s3://gfw2-data/analyses/umd_area_2013__from_gfw-data-lake/v1.10/raster/epsg-4326/10/40000/area_m/gdal-geotiff/"
pixel_area_zarr_name = "s3://gfw2-data/climate/AFOLU_flux_model/LULUCF/outputs/contextual_layer_global_zarr/20250604/global_pixel_area_20250604.zarr"

primary_forest_IFL_folder = "s3://gfw2-data/climate/carbon_model/ifl_primary_merged/processed/20200724/"
primary_forest_IFL_zarr_name = "s3://gfw2-data/climate/AFOLU_flux_model/LULUCF/outputs/contextual_layer_global_zarr/20250609/ifl_primary_forest_merged.zarr"


# Spreadsheet for state_node meanings
state_node_lookup_table_local = "/mnt/c/GIS/git/AFOLU_GHG_flux_model/src/LULUCF/LULUCF_state_node_lookup_table.xlsx"
state_node_lookup_table_s3 = "http://gfw2-data.s3.amazonaws.com/climate/AFOLU_flux_model/LULUCF/state_node_lookup_tables/LULUCF_state_node_lookup_table.xlsx"
sheet = "v030_20250430"

In [ ]:
# ## CREATES ZARRS FOR INPUTS NOT GENERATED BY THE AFOLU MODEL
# ## THIS SHOULD ONLY EVER HAVE TO BE DONE ONCE

# print(f"Reading inputs that apply to all intervals: {timestr()}")

# # GADM adm0
# adm0_uris = list_folder_uris(adm0_folder)
# print("adm0_folder:", adm0_folder)
# print(adm0_uris[0])
# print(f"Tile count in {adm0_folder}: {len(adm0_uris)}")

# print(f"   Reading adm0: {timestr()}")
# adm0_xarray_chunks = make_xarray_chunks(adm0_uris, chunk_size)
# adm0_xarray_chunks['band_data'] = adm0_xarray_chunks['band_data'].astype('uint16')  # adm0 should be uint16 but make_xarray_chunks makes it float64 for some reason
# print("adm0_xarray_chunks:", adm0_xarray_chunks)  # Print to confirm that the zarr datatype is correct

# print(f"   zarring adm0: {timestr()}")
# adm0_xarray_chunks.to_zarr(adm0_zarr_name, mode='w')
# print(f"   Finished zarring adm0: {timestr()}")


# # Pixel area
# pixel_area_uris = list_folder_uris(pixel_area_folder)
# print("pixel_area_folder:", pixel_area_folder)
# print(pixel_area_uris[0])
# print(f"Tile count in {pixel_area_folder}: {len(pixel_area_uris)}")

# print(f"   Reading pixel_area: {timestr()}")
# pixel_area_xarray_chunks = make_xarray_chunks(pixel_area_uris, chunk_size)
# print("pixel_area_xarray_chunks:", pixel_area_xarray_chunks)

# print(f"   zarring pixel_area: {timestr()}")
# pixel_area_xarray_chunks.to_zarr(pixel_area_zarr_name, mode='w')
# print(f"   Finished zarring pixel_area: {timestr()}")


# # Humid tropical primary forest/IFL merged
# primary_forest_IFL_uris = list_folder_uris(primary_forest_IFL_folder)
# print("primary_forest_IFL_folder:", primary_forest_IFL_folder)
# print(primary_forest_IFL_uris[0])
# print(f"Tile count in {primary_forest_IFL_folder}: {len(primary_forest_IFL_uris)}")

# print(f"   Reading primary_forest_IFL: {timestr()}")
# primary_forest_IFL_xarray_chunks = make_xarray_chunks(primary_forest_IFL_uris, chunk_size)
# primary_forest_IFL_xarray_chunks['band_data'] = primary_forest_IFL_xarray_chunks['band_data'].astype('uint8')  # should be uint8 but make_xarray_chunks makes it float32 for some reason
# print("primary_forest_IFL_xarray_chunks:", primary_forest_IFL_xarray_chunks)  # Print to confirm that the zarr datatype is correct

# print(f"   zarring primary_forest_IFL: {timestr()}")
# primary_forest_IFL_xarray_chunks.to_zarr(primary_forest_IFL_zarr_name, mode='w')
# print(f"   Finished zarring primary_forest_IFL: {timestr()}")

In [30]:
# Per a long exchange at https://chatgpt.com/g/g-vK4oPfjfp-coding-assistant/c/6846ff1c-4924-800a-97d2-705fd1033d99

def check_tile_shape(uri, expected_shape=EXPECTED_SHAPE):
    """
    Verifies that the raster at `uri` has the expected shape.
    """
    with rasterio.open(uri) as src:
        actual_shape = (src.height, src.width)
        # print(f"[Shape check] {uri} has shape {actual_shape}, expected {expected_shape}")
        if actual_shape != expected_shape:
            raise ValueError(f"{uri} shape {actual_shape} != expected {expected_shape}")

def preprocess_snap(ds):
    """
    Preprocesses a dataset by removing coordinate-affecting metadata and snapping
    its x/y coordinates to a canonical grid.
    """
    for attr in ["transform", "crs"]:
        ds.attrs.pop(attr, None)

    x0 = np.round(ds.x.values[0] / 0.00025) * 0.00025
    y0 = np.round(ds.y.values[0] / -0.00025) * -0.00025

    nx = ds.sizes["x"]
    ny = ds.sizes["y"]

    ds = ds.assign_coords({
        "x": x0 + np.arange(nx) * 0.00025,
        "y": y0 + np.arange(ny) * -0.00025
    })

    return ds

def open_and_reindex_tiles(tile_uris, chunk_size):
    """
    Loads, preprocesses, and reindexes tile URIs into a unified xarray dataset.
    """
    # Validate shapes first
    for uri in tile_uris:
        check_tile_shape(uri)

    # Open and snap coords
    ds = xr.open_mfdataset(
        sorted(tile_uris),
        combine="by_coords",
        parallel=True,
        chunks={'x': chunk_size, 'y': chunk_size},
        preprocess=preprocess_snap
    ).squeeze()

    # Canonical reindexing (restricted to existing coords)
    x_vals = ds.x.values
    y_vals = ds.y.values

    x_start = np.round(x_vals.min() / 0.00025) * 0.00025
    x_stop = np.round(x_vals.max() / 0.00025) * 0.00025
    y_start = np.round(y_vals.max() / 0.00025) * 0.00025
    y_stop = np.round(y_vals.min() / 0.00025) * 0.00025

    xs = np.arange(x_start, x_stop + 0.00025 / 2, 0.00025)
    ys = np.arange(y_start, y_stop - 0.00025 / 2, -0.00025)

    ds = ds.reindex(
        x=xs, y=ys,
        method="nearest",
        tolerance=1e-6  # adjust as needed to match your snapping precision
    )

    return ds

In [49]:
%%time

### CONVERTS GEOTIFS TO ZARRS AND STORES THEM IN S3
### ONLY NEED TO DO THE FIRST TIME RUNNING AN ANALYSIS ON MODEL OUTPUTS

analysis_start_time = time.time()

for interval_end_year in interval_end_years:

    interval = f"{interval_end_year-1}_{interval_end_year}"

    print(f"Processing {interval}: {timestr()}")
    interval_start_time = time.time()

    # # Creates a Pandas series of s3 uris for this specific analysis layer
    # gross_emis_CO2_folder_interval = gross_emis_CO2_folder.replace("INTERVAL", interval)
    # gross_emis_CO2_uris = list_folder_uris(gross_emis_CO2_folder_interval)

    # gross_emis_all_gases_folder_interval = gross_emis_all_gases_folder.replace("INTERVAL", interval)
    # gross_emis_all_gases_uris = list_folder_uris(gross_emis_all_gases_folder_interval)
    
    # gross_remv_all_pools_folder_interval = gross_remv_all_pools_folder.replace("INTERVAL", interval)
    # gross_remv_all_pools_uris = list_folder_uris(gross_remv_all_pools_folder_interval)
    
    # net_flux_all_pools_CO2_folder_interval = net_flux_all_pools_CO2_folder.replace("INTERVAL", interval)
    # net_flux_all_pools_CO2_uris = list_folder_uris(net_flux_all_pools_CO2_folder_interval)
    
    node_folder_interval = node_folder.replace("INTERVAL", interval)
    node_tile_year_uris = list_folder_uris(node_folder_interval)

    
    # print("    gross_emis_CO2_folder_interval:", gross_emis_CO2_folder_interval)
    # print(gross_emis_CO2_uris[0])
    # print(f"    Tile count in {gross_emis_CO2_folder_interval}: {len(gross_emis_CO2_uris)}")
    
    # print("    gross_emis_all_gases_folder_interval:", gross_emis_all_gases_folder_interval)
    # print(gross_emis_all_gases_uris[0])
    # print(f"    Tile count in {gross_emis_all_gases_folder_interval}: {len(gross_emis_all_gases_uris)}")
    
    # print("    gross_remv_all_pools_folder_interval:", gross_remv_all_pools_folder_interval)
    # print(gross_emis_CO2_uris[0])
    # print(f"    Tile count in {gross_remv_all_pools_folder_interval}: {len(gross_emis_CO2_uris)}")

    # print("    gross_emis_CO2_folder_interval:", net_flux_all_pools_CO2_folder_interval)
    # print(gross_remv_all_pools_uris[0])
    # print(f"    Tile count in {net_flux_all_pools_CO2_folder_interval}: {len(gross_remv_all_pools_uris)}")

    print("    node_folder_interval:", node_folder_interval)
    print(node_tile_year_uris[0])
    print(f"    Tile count in {node_folder_interval}: {len(node_tile_year_uris)}")
    
    # print(f"   Reading gross emis CO2 only for {interval}: {timestr()}")
    # gross_emis_CO2_xarray_chunks = open_and_reindex_tiles(gross_emis_CO2_uris, chunk_size=40000)
    # print(f"   Reading gross emis all gases for {interval}: {timestr()}")
    # gross_emis_all_gases_xarray_chunks = open_and_reindex_tiles(gross_emis_all_gases_uris, chunk_size=40000) 
    # print(f"   Reading gross removals for {interval}: {timestr()}")
    # gross_remv_all_pools_xarray_chunks = open_and_reindex_tiles(gross_remv_all_pools_uris, chunk_size=40000) 
    # print(f"   Reading net flux CO2 only for {interval}: {timestr()}")
    # net_flux_all_pools_CO2_xarray_chunks = open_and_reindex_tiles(net_flux_all_pools_CO2_uris, chunk_size=40000)
    print(f"   Reading state_nodes for {interval}: {timestr()}")
    node_xarray_chunks = open_and_reindex_tiles(node_tile_year_uris, chunk_size=10000)
    node_xarray_chunks['band_data'] = node_xarray_chunks['band_data'].astype('uint32')  # state_nodes should be uint32 but make_xarray_chunks makes it float64 for some reason

    analysis_layers = [
        # gross_emis_CO2_xarray_chunks,
        # gross_emis_all_gases_xarray_chunks,
        # gross_remv_all_pools_xarray_chunks,
        # net_flux_all_pools_CO2_xarray_chunks,
        node_xarray_chunks
    ]

    for ds in analysis_layers:
        print(ds)
        print("X resolution:", np.diff(ds.x.values).mean())
        print("Y resolution:", np.diff(ds.y.values).mean())
        print("X range:", ds.x.values[[0, -1]])
        print("Y range:", ds.y.values[[0, -1]])
        print("Shape (y, x):", ds.sizes['y'], ds.sizes['x'])


    persisted_nodes = node_xarray_chunks.persist()
        
    # # per https://chatgpt.com/g/g-vK4oPfjfp-coding-assistant/c/68309b36-0f48-800a-bd56-67180b55106e
    # gross_emis_CO2_zarr_name = f"{zarr_s3_path}{interval}/gross_emissions__all_C_pools__CO2_only__MgCO2_{interval}.zarr"
    # gross_emis_all_gases_zarr_name = f"{zarr_s3_path}{interval}/gross_emissions__all_C_pools__all_gases__MgCO2e_pixel_yr_{interval}.zarr"
    # gross_remv_all_pools_zarr_name = f"{zarr_s3_path}{interval}/gross_removals__all_C_pools__MgCO2_pixel_yr_{interval}.zarr"
    # net_flux_all_pools_CO2_zarr_name = f"{zarr_s3_path}{interval}/net_flux__all_C_pools__CO2_only__MgCO2_pixel_yr_{interval}.zarr"
    node_zarr_name = f"{zarr_s3_path}{interval}/land_state_node_{interval}.zarr"
    
    # print(f"   zarring gross emis CO2 only for {interval}: {timestr()}")
    # gross_emis_CO2_xarray_chunks.to_zarr(gross_emis_CO2_zarr_name, mode='w')
    # print(f"   zarring gross emis all gases only for {interval}: {timestr()}")
    # gross_emis_all_gases_xarray_chunks.to_zarr(gross_emis_all_gases_zarr_name, mode='w')
    # print(f"   zarring gross removals for {interval}: {timestr()}")
    # gross_remv_all_pools_xarray_chunks.to_zarr(gross_remv_all_pools_zarr_name)
    # print(f"   zarring net flux CO2 only for {interval}: {timestr()}")
    # net_flux_all_pools_CO2_xarray_chunks.to_zarr(net_flux_all_pools_CO2_zarr_name, mode='w')
    print(f"   zarring state_nodes for {interval}: {timestr()}")
    persisted_nodes.to_zarr(node_zarr_name, mode='w')

    print(f"    Done zarring: {timestr()}")
    interval_end_time = time.time()
    print(f"   {interval} took {round(interval_end_time - interval_start_time)} seconds")

print(f"Done zarring all intervals: {timestr()}")
analysis_end_time = time.time()
print(f"Analysis took {round(analysis_end_time - analysis_start_time)} seconds")

Processing 2015_2016: 20250609_16_02_38
    node_folder_interval: s3://gfw2-data/climate/AFOLU_flux_model/LULUCF/outputs/version_0_3_3/land_state_node/standard_model/annual_intervals/2015_2016/40000_pixels/20250511/
s3://gfw2-data/climate/AFOLU_flux_model/LULUCF/outputs/version_0_3_3/land_state_node/standard_model/annual_intervals/2015_2016/40000_pixels/20250511/00N_010E__land_state_node_2015_2016.tif
    Tile count in s3://gfw2-data/climate/AFOLU_flux_model/LULUCF/outputs/version_0_3_3/land_state_node/standard_model/annual_intervals/2015_2016/40000_pixels/20250511/: 27
   Reading state_nodes for 2015_2016: 20250609_16_02_38
<xarray.Dataset> Size: 851GB
Dimensions:      (y: 280000, x: 760000)
Coordinates:
  * y            (y) float64 2MB 50.0 50.0 50.0 50.0 ... -20.0 -20.0 -20.0 -20.0
  * x            (x) float64 6MB -40.0 -40.0 -40.0 -40.0 ... 150.0 150.0 150.0
    band         int64 8B 1
    spatial_ref  int64 8B 0
Data variables:
    band_data    (y, x) uint32 851GB dask.array<chunk

AttributeError: 'tuple' object has no attribute 'size'

In [45]:
print(node_xarray_chunks)

<xarray.Dataset> Size: 851GB
Dimensions:      (y: 280000, x: 760000)
Coordinates:
  * y            (y) float64 2MB 50.0 50.0 50.0 50.0 ... -20.0 -20.0 -20.0 -20.0
  * x            (x) float64 6MB -40.0 -40.0 -40.0 -40.0 ... 150.0 150.0 150.0
    band         int64 8B 1
    spatial_ref  int64 8B 0
Data variables:
    band_data    (y, x) uint32 851GB dask.array<chunksize=(10000, 10000), meta=np.ndarray>


In [56]:
%%time

combined_df = pd.DataFrame()
analysis_start_time = time.time()

state_node_df = create_state_node_df(state_node_lookup_table_local, state_node_lookup_table_s3, sheet)
# print(state_node_df)

print("Opening zarrs for non-annual inputs")
pixel_area = xr.open_zarr(pixel_area_zarr_name).band_data
adm0 = xr.open_zarr(adm0_zarr_name).band_data
primary_forest_IFL = xr.open_zarr(primary_forest_IFL_zarr_name).band_data


for interval_end_year in interval_end_years:

    interval = f"{interval_end_year-1}_{interval_end_year}"

    print(f"Processing {interval}: {timestr()}")
    interval_start_time = time.time()

    # Creates a Pandas series of s3 uris for this specific analysis layer
    gross_emis_CO2_folder_interval = gross_emis_CO2_folder.replace("INTERVAL", interval)
    gross_emis_CO2_uris = list_folder_uris(gross_emis_CO2_folder_interval)
    gross_emis_all_gases_folder_interval = gross_emis_all_gases_folder.replace("INTERVAL", interval)
    gross_emis_all_gases_uris = list_folder_uris(gross_emis_all_gases_folder_interval)
    gross_remv_all_pools_folder_interval = gross_remv_all_pools_folder.replace("INTERVAL", interval)
    gross_remv_all_pools_uris = list_folder_uris(gross_remv_all_pools_folder_interval)
    net_flux_all_pools_CO2_folder_interval = net_flux_all_pools_CO2_folder.replace("INTERVAL", interval)
    net_flux_all_pools_CO2_uris = list_folder_uris(net_flux_all_pools_CO2_folder_interval)
    node_folder_interval = node_folder.replace("INTERVAL", interval)
    node_tile_year_uris = list_folder_uris(node_folder_interval)

    # Gets input layer metadata, like the output pattern.
    gross_emis_CO2_output_pattern = parse_pattern_from_uri(gross_emis_CO2_uris)
    gross_emis_all_gases_output_pattern = parse_pattern_from_uri(gross_emis_all_gases_uris)
    gross_remv_all_pools_output_pattern = parse_pattern_from_uri(gross_remv_all_pools_uris)
    net_flux_output_pattern = parse_pattern_from_uri(net_flux_all_pools_CO2_uris)
    node_output_pattern = parse_pattern_from_uri(node_tile_year_uris)

    # # Version if not using zarrs for inputs. Reads the geotifs into chunks directly. This is slower than creating zarrs and then reading from them, but leaving it here for reference. 
    # print(f"   Reading gross emis CO2 only for {interval}: {timestr()}")
    # gross_emis_CO2 = make_xarray_chunks(gross_emis_CO2_uris, chunk_size)['band_data']
    # print(f"   Reading gross emis all gases for {interval}: {timestr()}")
    # gross_emis_all_gases = make_xarray_chunks(gross_emis_all_gases_uris, chunk_size)['band_data']
    # print(f"   Reading gross removals for {interval}: {timestr()}")
    # gross_remv_all_pools = make_xarray_chunks(gross_remv_all_pools_uris, chunk_size)['band_data']   
    # print(f"   Reading net flux CO2 only for {interval}: {timestr()}")
    # net_flux_all_pools_CO2 = make_xarray_chunks(net_flux_all_pools_CO2_uris, chunk_size)['band_data']   
    # print(f"   Reading state_nodes for {interval}: {timestr()}")
    # state_nodes = make_xarray_chunks(node_tile_year_uris, chunk_size)['band_data']  
    # state_nodes = state_nodes.astype('uint32')  # state_nodes should be uint32 but make_xarray_chunks makes it float64 for some reason

    # Locations of zarrs to be read
    gross_emis_CO2_zarr_name = f"{zarr_s3_path}{interval}/gross_emissions__all_C_pools__CO2_only__MgCO2_{interval}.zarr"
    gross_emis_all_gases_zarr_name = f"{zarr_s3_path}{interval}/gross_emissions__all_C_pools__all_gases__MgCO2e_pixel_yr_{interval}.zarr"
    gross_remv_all_pools_zarr_name = f"{zarr_s3_path}{interval}/gross_removals__all_C_pools__MgCO2_pixel_yr_{interval}.zarr"
    net_flux_all_pools_CO2_zarr_name = f"{zarr_s3_path}{interval}/net_flux__all_C_pools__CO2_only__MgCO2_pixel_yr_{interval}.zarr"
    node_zarr_name = f"{zarr_s3_path}{interval}/land_state_node_{interval}.zarr"

    # Reads zarrs
    print(f"   Reading zar for gross emis CO2 only for {interval}: {timestr()}")
    gross_emis_CO2 = xr.open_zarr(gross_emis_CO2_zarr_name).band_data
    print(f"   Reading zar for gross emis all gases only for {interval}: {timestr()}")
    gross_emis_all_gases = xr.open_zarr(gross_emis_all_gases_zarr_name).band_data
    print(f"   Reading zar for gross removals for {interval}: {timestr()}")
    gross_remv_all_pools= xr.open_zarr(gross_remv_all_pools_zarr_name).band_data
    print(f"   Reading zar for net flux CO2 only for {interval}: {timestr()}")
    net_flux_all_pools_CO2 = xr.open_zarr(net_flux_all_pools_CO2_zarr_name).band_data
    print(f"   Reading zar for nodes for {interval}: {timestr()}")
    state_nodes = xr.open_zarr(node_zarr_name).band_data

    # Makes all inputs align with the state_node extent (i.e. analysis limited to state_node extent, not extent of e.g., GADM)
    print(f"   Aligning {interval}: {timestr()}")
    state_nodes_aligned, pixel_area_aligned = xr.align(state_nodes, pixel_area, join="inner")
    state_nodes_aligned, adm0_aligned = xr.align(state_nodes, adm0, join="inner")
        
    # Each contextual layer has to have a unique name.  
    # Necessary to keep flox from getting confused about having multiple band_data to work with, per Solomon in Slack (2025-06-04). 
    # Any other contextual layers need to be renamed here, too. 
    adm0_aligned.name = "gadm_adm0"
    state_nodes_aligned.name = "state_nodes"


    
    # # Makes all inputs align with the state_node extent (i.e. analysis limited to state_node extent, not extent of e.g., GADM)
    # print(f"   Aligning {interval}: {timestr()}")
    # state_nodes_aligned, pixel_area_aligned = xr.align(state_nodes, pixel_area, join="inner")
    # state_nodes_aligned, adm0_aligned = xr.align(state_nodes, adm0, join="inner")
    # # pixel_area_aligned, _  = xr.align(pixel_area, state_nodes, join="inner")
    # # adm0_aligned, _ = xr.align(adm0, state_nodes, join="inner")
    # print("state_nodes:", state_nodes)
    # print("primary_forest_IFL:", primary_forest_IFL)
    primary_forest_IFL_aligned = primary_forest_IFL.reindex_like(state_nodes_aligned, method=None).chunk({'y': 10000, 'x': 10000})
    # # print("state_nodes:", state_nodes)
    # # print("primary_forest_IFL_aligned:", primary_forest_IFL_aligned)
        
    # # Each contextual layer has to have a unique name.  
    # # Necessary to keep flox from getting confused about having multiple band_data to work with, per Solomon in Slack (2025-06-04). 
    # # Any other contextual layers need to be renamed here, too. 
    # adm0_aligned.name = "gadm_adm0"
    # state_nodes.name = "state_nodes"
    primary_forest_IFL_aligned.name = "primary_forest_IFL"

    sys.quit()


    # Makes a datacube of all the analysis layers.
    # Add any other analysis layers here.
    print(f"   Stacking {interval}: {timestr()}")
    flux_cube = xr.DataArray(dask.array.stack((
                                               gross_emis_CO2, 
                                               gross_emis_all_gases, 
                                               gross_remv_all_pools, 
                                               net_flux_all_pools_CO2, 
                                               pixel_area_aligned)), 
                             dims=('flux_type', 'y', 'x'))
    print("flux_cube:", flux_cube)

    # print(f"   Computing {interval}: {timestr()}")
    # flux_results = xarray_reduce(
    #     flux_cube,  # Layers to be analyzed
    #     # pixel_area_aligned,  # to test pixel_area as the only analysis layer
    #     *(adm0_aligned, state_nodes, primary_forest_IFL_aligned),  # Contextual layers
    #     func='sum',
    #     expected_groups=(gadm_adm0_ids, node_codes, primary_forest_IFL_codes),  # Contextual layer possible values. Must be in some order as contextual layers above. 
    #     reindex=ReindexStrategy(
    #         blockwise=False,
    #         array_type=ReindexArrayType.SPARSE_COO
    #     ),
    #     fill_value=0
    # ).compute()

    print(f"   Computing {interval}: {timestr()}")
    flux_results = xarray_reduce(
        flux_cube,  # Layers to be analyzed
        # pixel_area_aligned,  # to test pixel_area as the only analysis layer
        primary_forest_IFL_aligned,  # Contextual layers
        func='sum',
        expected_groups=(primary_forest_IFL_codes),  # Contextual layer possible values. Must be in some order as contextual layers above. 
        reindex=ReindexStrategy(
            blockwise=False,
            array_type=ReindexArrayType.SPARSE_COO
        ),
        fill_value=0
    ).compute()


    # print(f"   Computing {interval}: {timestr()}")
    # flux_results = xarray_reduce(
    #     flux_cube,  # Layers to be analyzed
    #     # pixel_area_aligned,  # to test pixel_area as the only analysis layer
    #     *(adm0_aligned, state_nodes_aligned),  # Contextual layers
    #     func='sum',
    #     expected_groups=(gadm_adm0_ids, node_codes),  # Contextual layer possible values. Must be in some order as contextual layers above. 
    #     reindex=ReindexStrategy(
    #         blockwise=False,
    #         array_type=ReindexArrayType.SPARSE_COO
    #     ),
    #     fill_value=0
    # ).compute()

    # Prepared outputs for conversion into dataframe
    coord_dict = convert_to_coord_dict(flux_results)
    
    # Creates the dataframe for the interval and does some processing of it
    df = create_interval_df(coord_dict, state_node_df)
    # print(df)

    df = calculate_interval_flux_densities(df)
    # print(df)

    # Combines dataframe from this interval with dataframes from previous intervals
    combined_df = pd.concat([combined_df, df])

    interval_end_time = time.time()
    print(f"   {interval} took {round(interval_end_time - interval_start_time)} seconds")


combined_df = combined_df.reset_index(drop=True)

analysis_end_time = time.time()
print(f"Analysis took {round(analysis_end_time - analysis_start_time)} seconds")

print(combined_df)

Opening zarrs for non-annual inputs
Processing 2015_2016: 20250609_13_32_38
   Reading zar for gross emis CO2 only for 2015_2016: 20250609_13_32_38
   Reading zar for gross emis all gases only for 2015_2016: 20250609_13_32_39
   Reading zar for gross removals for 2015_2016: 20250609_13_32_39
   Reading zar for net flux CO2 only for 2015_2016: 20250609_13_32_39
   Reading zar for nodes for 2015_2016: 20250609_13_32_40
   Aligning 2015_2016: 20250609_13_32_40


/home/dagibbs22/miniforge3/envs/coiled_20250606/lib/python3.12/site-packages/dask/array/core.py:5092: PerformanceWarning: Increasing number of chunks by factor of 24
  result = blockwise(


NameError: name 'sys' is not defined

In [57]:
print("Shape:")
print("  state_nodes:", state_nodes.shape)
print("  primary_forest_IFL:", primary_forest_IFL.shape)

print("Coordinate lengths:")
print("  state_nodes.x:", len(state_nodes.x))
print("  primary_forest_IFL.x:", len(primary_forest_IFL.x))
print("  state_nodes.y:", len(state_nodes.y))
print("  primary_forest_IFL.y:", len(primary_forest_IFL.y))

Shape:
  state_nodes: (240000, 560000)
  primary_forest_IFL: (520000, 1360000)
Coordinate lengths:
  state_nodes.x: 560000
  primary_forest_IFL.x: 1360000
  state_nodes.y: 240000
  primary_forest_IFL.y: 520000


In [58]:
print("First 5 x values:")
print("  state_nodes:", state_nodes.x.values[:5])
print("  primary_forest_IFL:", primary_forest_IFL.x.values[:5])

print("Last 5 x values:")
print("  state_nodes:", state_nodes.x.values[-5:])
print("  primary_forest_IFL:", primary_forest_IFL.x.values[-5:])

First 5 x values:
  state_nodes: [-39.999875 -39.999625 -39.999375 -39.999125 -39.998875]
  primary_forest_IFL: [-169.999875 -169.999625 -169.999375 -169.999125 -169.998875]
Last 5 x values:
  state_nodes: [149.998875 149.999125 149.999375 149.999625 149.999875]
  primary_forest_IFL: [179.998875 179.999125 179.999375 179.999625 179.999875]


In [59]:
print("First 5 y values:")
print("  state_nodes:", state_nodes.y.values[:5])
print("  primary_forest_IFL:", primary_forest_IFL.y.values[:5])

print("Last 5 y values:")
print("  state_nodes:", state_nodes.y.values[-5:])
print("  primary_forest_IFL:", primary_forest_IFL.y.values[-5:])

First 5 y values:
  state_nodes: [49.999875 49.999625 49.999375 49.999125 49.998875]
  primary_forest_IFL: [69.999875 69.999625 69.999375 69.999125 69.998875]
Last 5 y values:
  state_nodes: [-19.998875 -19.999125 -19.999375 -19.999625 -19.999875]
  primary_forest_IFL: [-59.998875 -59.999125 -59.999375 -59.999625 -59.999875]


In [62]:
print("X resolution:")
print("  pixel_area:", np.diff(pixel_area.x.values).mean())
print("  gross_emis_CO2:", np.diff(gross_emis_CO2.x.values).mean())
print("  state_nodes:", np.diff(state_nodes.x.values).mean())
print("  primary_forest_IFL:", np.diff(primary_forest_IFL.x.values).mean())

print("Y resolution:")
print("  pixel_area:", np.diff(pixel_area.y.values).mean())
print("  gross_emis_CO2:", np.diff(gross_emis_CO2.y.values).mean())
print("  state_nodes:", np.diff(state_nodes.y.values).mean())
print("  primary_forest_IFL:", np.diff(primary_forest_IFL.y.values).mean())

X resolution:
  pixel_area: 0.0002500000000000001
  gross_emis_CO2: 0.00033928587372477465
  state_nodes: 0.00033928587372477465
  primary_forest_IFL: 0.0002573529465830491
Y resolution:
  pixel_area: -0.0002500000000000001
  gross_emis_CO2: -0.0002916668402785013
  state_nodes: -0.0002916668402785013
  primary_forest_IFL: -0.00025000000000000017


In [ ]:
print("x matches:", np.allclose(state_nodes.x.values, primary_forest_IFL.x.values))
print("y matches:", np.allclose(state_nodes.y.values, primary_forest_IFL.y.values))

In [64]:
print("X range:")
print("  state_nodes:", (state_nodes.x.min().item(), state_nodes.x.max().item()))
print("  primary_forest_IFL:", (primary_forest_IFL.x.min().item(), primary_forest_IFL.x.max().item()))

print("Y range:")
print("  state_nodes:", (state_nodes.y.min().item(), state_nodes.y.max().item()))
print("  primary_forest_IFL:", (primary_forest_IFL.y.min().item(), primary_forest_IFL.y.max().item()))


X range:
  state_nodes: (-39.999875, 149.999875)
  primary_forest_IFL: (-169.999875, 179.999875)
Y range:
  state_nodes: (-19.999875000000003, 49.999875)
  primary_forest_IFL: (-59.999874999999996, 69.999875)


In [ ]:
# combined_df[(combined_df.flux_type == gross_remv_all_pools_output_pattern) 
# & (combined_df.gadm_adm0 == 180)]
combined_df[(combined_df.state_node == 'n3222121')]

In [ ]:
combined_df[(combined_df.flux_type == net_flux_output_pattern) 
& (combined_df.state_node == 'n1110000')]
# combined_df[(combined_df.state_node == 'n2112000')]
# combined_df.groupby(combined_df.gadm_adm0).sum()

In [ ]:
result_df[(result_df.flux_type == f'{gross_remv_all_pools_output_pattern}__per_ha') 
& (result_df.state_node == 'n5100000')]

In [ ]:
combined_df_wide = combined_df.pivot(index=['state_node', 'interval_end', 'node_grp', 'gadm_adm0', 'meaning'], columns="flux_type", values="value").reset_index()
# combined_df_wide

In [ ]:
# # Export to a csv so data can be used in Excel or reused
# combined_df_wide.to_csv('DRC_2016_2023.csv', index=False)

In [ ]:
walker = pyg.walk(combined_df_wide)

In [ ]:
vis_spec = r"""{"config":[{"config":{"defaultAggregated":true,"geoms":["line"],"coordSystem":"generic","limit":-1},"encodings":{"dimensions":[{"dragId":"gw_5Dux","fid":"state_node","name":"state_node","basename":"state_node","semanticType":"nominal","analyticType":"dimension"},{"dragId":"gw__GB7","fid":"interval_end","name":"interval_end","basename":"interval_end","semanticType":"ordinal","analyticType":"dimension"},{"dragId":"gw_Xb3-","fid":"node_grp","name":"node_grp","basename":"node_grp","semanticType":"nominal","analyticType":"dimension"},{"dragId":"gw_IGtN","fid":"gadm_adm0","name":"gadm_adm0","basename":"gadm_adm0","semanticType":"quantitative","analyticType":"dimension"},{"dragId":"gw_848c","fid":"meaning","name":"meaning","basename":"meaning","semanticType":"nominal","analyticType":"dimension"},{"dragId":"gw_mea_key_fid","fid":"gw_mea_key_fid","name":"Measure names","analyticType":"dimension","semanticType":"nominal"}],"measures":[{"dragId":"gw_y99d","fid":"area__ha","name":"area__ha","basename":"area__ha","analyticType":"measure","semanticType":"quantitative","aggName":"sum"},{"dragId":"gw_Sjqt","fid":"gross_emissions__all_C_pools__CO2_only__MgCO2","name":"gross_emissions__all_C_pools__CO2_only__MgCO2","basename":"gross_emissions__all_C_pools__CO2_only__MgCO2","analyticType":"measure","semanticType":"quantitative","aggName":"sum"},{"dragId":"gw_5GlR","fid":"gross_emissions__all_C_pools__CO2_only__MgCO2__C_per_ha","name":"gross_emissions__all_C_pools__CO2_only__MgCO2__C_per_ha","basename":"gross_emissions__all_C_pools__CO2_only__MgCO2__C_per_ha","analyticType":"measure","semanticType":"quantitative","aggName":"sum"},{"dragId":"gw_axS0","fid":"gross_emissions__all_C_pools__all_gases__MgCO2e","name":"gross_emissions__all_C_pools__all_gases__MgCO2e","basename":"gross_emissions__all_C_pools__all_gases__MgCO2e","analyticType":"measure","semanticType":"quantitative","aggName":"sum"},{"dragId":"gw_GZnF","fid":"gross_emissions__all_C_pools__all_gases__MgCO2e__C_per_ha","name":"gross_emissions__all_C_pools__all_gases__MgCO2e__C_per_ha","basename":"gross_emissions__all_C_pools__all_gases__MgCO2e__C_per_ha","analyticType":"measure","semanticType":"quantitative","aggName":"sum"},{"dragId":"gw_cuXy","fid":"gross_removals__all_C_pools__MgCO2","name":"gross_removals__all_C_pools__MgCO2","basename":"gross_removals__all_C_pools__MgCO2","analyticType":"measure","semanticType":"quantitative","aggName":"sum"},{"dragId":"gw_t1MS","fid":"gross_removals__all_C_pools__MgCO2__C_per_ha","name":"gross_removals__all_C_pools__MgCO2__C_per_ha","basename":"gross_removals__all_C_pools__MgCO2__C_per_ha","analyticType":"measure","semanticType":"quantitative","aggName":"sum"},{"dragId":"gw_12Yi","fid":"net_flux__all_C_pools__CO2_only__MgCO2","name":"net_flux__all_C_pools__CO2_only__MgCO2","basename":"net_flux__all_C_pools__CO2_only__MgCO2","analyticType":"measure","semanticType":"quantitative","aggName":"sum"},{"dragId":"gw_e6y_","fid":"net_flux__all_C_pools__CO2_only__MgCO2__C_per_ha","name":"net_flux__all_C_pools__CO2_only__MgCO2__C_per_ha","basename":"net_flux__all_C_pools__CO2_only__MgCO2__C_per_ha","analyticType":"measure","semanticType":"quantitative","aggName":"sum"},{"dragId":"gw_count_fid","fid":"gw_count_fid","name":"Row count","analyticType":"measure","semanticType":"quantitative","aggName":"sum","computed":true,"expression":{"op":"one","params":[],"as":"gw_count_fid"}},{"dragId":"gw_mea_val_fid","fid":"gw_mea_val_fid","name":"Measure values","analyticType":"measure","semanticType":"quantitative","aggName":"sum"}],"rows":[{"dragId":"gw_8-h6","fid":"gross_emissions__all_C_pools__CO2_only__MgCO2","name":"gross_emissions__all_C_pools__CO2_only__MgCO2","basename":"gross_emissions__all_C_pools__CO2_only__MgCO2","analyticType":"measure","semanticType":"quantitative","aggName":"sum"}],"columns":[{"dragId":"gw_uU2i","fid":"interval_end","name":"interval_end","basename":"interval_end","semanticType":"ordinal","analyticType":"dimension"}],"color":[{"dragId":"gw_DITZ","fid":"meaning","name":"meaning","basename":"meaning","semanticType":"nominal","analyticType":"dimension"}],"opacity":[],"size":[],"shape":[],"radius":[],"theta":[],"longitude":[],"latitude":[],"geoId":[],"details":[],"filters":[{"dragId":"gw_GO_f","fid":"gadm_adm0","name":"gadm_adm0","basename":"gadm_adm0","semanticType":"quantitative","analyticType":"dimension","rule":{"type":"one of","value":[180]}},{"dragId":"gw_c6Cy","fid":"meaning","name":"meaning","basename":"meaning","semanticType":"nominal","analyticType":"dimension","rule":{"type":"one of","value":["<100 year old natural forest not disturbed in last interval, with fire",">100 year old natural forest not disturbed in last interval, with fire","Natural forest converted to short vegetation with disturbance that emits all non-soil C pools, without fire","Forest partially disturbed in the last interval without signif. height increase after, without fire"]}}],"text":[]},"layout":{"showActions":false,"showTableSummary":false,"stack":"stack","interactiveScale":false,"zeroScale":true,"size":{"mode":"auto","width":320,"height":200},"format":{},"geoKey":"name","resolve":{"x":false,"y":false,"color":false,"opacity":false,"shape":false,"size":false}},"visId":"gw_wEzF","name":"Emissions"},{"config":{"defaultAggregated":true,"geoms":["line"],"coordSystem":"generic","limit":-1},"encodings":{"dimensions":[{"dragId":"gw_5Dux","fid":"state_node","name":"state_node","basename":"state_node","semanticType":"nominal","analyticType":"dimension"},{"dragId":"gw__GB7","fid":"interval_end","name":"interval_end","basename":"interval_end","semanticType":"ordinal","analyticType":"dimension"},{"dragId":"gw_Xb3-","fid":"node_grp","name":"node_grp","basename":"node_grp","semanticType":"nominal","analyticType":"dimension"},{"dragId":"gw_IGtN","fid":"gadm_adm0","name":"gadm_adm0","basename":"gadm_adm0","semanticType":"quantitative","analyticType":"dimension"},{"dragId":"gw_848c","fid":"meaning","name":"meaning","basename":"meaning","semanticType":"nominal","analyticType":"dimension"},{"dragId":"gw_mea_key_fid","fid":"gw_mea_key_fid","name":"Measure names","analyticType":"dimension","semanticType":"nominal"}],"measures":[{"dragId":"gw_y99d","fid":"area__ha","name":"area__ha","basename":"area__ha","analyticType":"measure","semanticType":"quantitative","aggName":"sum"},{"dragId":"gw_Sjqt","fid":"gross_emissions__all_C_pools__CO2_only__MgCO2","name":"gross_emissions__all_C_pools__CO2_only__MgCO2","basename":"gross_emissions__all_C_pools__CO2_only__MgCO2","analyticType":"measure","semanticType":"quantitative","aggName":"sum"},{"dragId":"gw_5GlR","fid":"gross_emissions__all_C_pools__CO2_only__MgCO2__C_per_ha","name":"gross_emissions__all_C_pools__CO2_only__MgCO2__C_per_ha","basename":"gross_emissions__all_C_pools__CO2_only__MgCO2__C_per_ha","analyticType":"measure","semanticType":"quantitative","aggName":"sum"},{"dragId":"gw_axS0","fid":"gross_emissions__all_C_pools__all_gases__MgCO2e","name":"gross_emissions__all_C_pools__all_gases__MgCO2e","basename":"gross_emissions__all_C_pools__all_gases__MgCO2e","analyticType":"measure","semanticType":"quantitative","aggName":"sum"},{"dragId":"gw_GZnF","fid":"gross_emissions__all_C_pools__all_gases__MgCO2e__C_per_ha","name":"gross_emissions__all_C_pools__all_gases__MgCO2e__C_per_ha","basename":"gross_emissions__all_C_pools__all_gases__MgCO2e__C_per_ha","analyticType":"measure","semanticType":"quantitative","aggName":"sum"},{"dragId":"gw_cuXy","fid":"gross_removals__all_C_pools__MgCO2","name":"gross_removals__all_C_pools__MgCO2","basename":"gross_removals__all_C_pools__MgCO2","analyticType":"measure","semanticType":"quantitative","aggName":"sum"},{"dragId":"gw_t1MS","fid":"gross_removals__all_C_pools__MgCO2__C_per_ha","name":"gross_removals__all_C_pools__MgCO2__C_per_ha","basename":"gross_removals__all_C_pools__MgCO2__C_per_ha","analyticType":"measure","semanticType":"quantitative","aggName":"sum"},{"dragId":"gw_12Yi","fid":"net_flux__all_C_pools__CO2_only__MgCO2","name":"net_flux__all_C_pools__CO2_only__MgCO2","basename":"net_flux__all_C_pools__CO2_only__MgCO2","analyticType":"measure","semanticType":"quantitative","aggName":"sum"},{"dragId":"gw_e6y_","fid":"net_flux__all_C_pools__CO2_only__MgCO2__C_per_ha","name":"net_flux__all_C_pools__CO2_only__MgCO2__C_per_ha","basename":"net_flux__all_C_pools__CO2_only__MgCO2__C_per_ha","analyticType":"measure","semanticType":"quantitative","aggName":"sum"},{"dragId":"gw_count_fid","fid":"gw_count_fid","name":"Row count","analyticType":"measure","semanticType":"quantitative","aggName":"sum","computed":true,"expression":{"op":"one","params":[],"as":"gw_count_fid"}},{"dragId":"gw_mea_val_fid","fid":"gw_mea_val_fid","name":"Measure values","analyticType":"measure","semanticType":"quantitative","aggName":"sum"}],"rows":[{"dragId":"gw_fa0j","fid":"gross_emissions__all_C_pools__CO2_only__MgCO2__C_per_ha","name":"gross_emissions__all_C_pools__CO2_only__MgCO2__C_per_ha","basename":"gross_emissions__all_C_pools__CO2_only__MgCO2__C_per_ha","analyticType":"measure","semanticType":"quantitative","aggName":"sum"}],"columns":[{"dragId":"gw_uU2i","fid":"interval_end","name":"interval_end","basename":"interval_end","semanticType":"ordinal","analyticType":"dimension"}],"color":[{"dragId":"gw_DITZ","fid":"meaning","name":"meaning","basename":"meaning","semanticType":"nominal","analyticType":"dimension"}],"opacity":[],"size":[],"shape":[],"radius":[],"theta":[],"longitude":[],"latitude":[],"geoId":[],"details":[],"filters":[{"dragId":"gw_GO_f","fid":"gadm_adm0","name":"gadm_adm0","basename":"gadm_adm0","semanticType":"quantitative","analyticType":"dimension","rule":{"type":"one of","value":[180]}},{"dragId":"gw_c6Cy","fid":"meaning","name":"meaning","basename":"meaning","semanticType":"nominal","analyticType":"dimension","rule":{"type":"one of","value":["<100 year old natural forest not disturbed in last interval, with fire",">100 year old natural forest not disturbed in last interval, with fire","Natural forest converted to short vegetation with disturbance that emits all non-soil C pools, without fire","Forest partially disturbed in the last interval without signif. height increase after, without fire"]}}],"text":[]},"layout":{"showActions":false,"showTableSummary":false,"stack":"stack","interactiveScale":false,"zeroScale":true,"size":{"mode":"auto","width":320,"height":200},"format":{},"geoKey":"name","resolve":{"x":false,"y":false,"color":false,"opacity":false,"shape":false,"size":false}},"visId":"gw_CA7R","name":"Emission factor"},{"config":{"defaultAggregated":true,"geoms":["line"],"coordSystem":"generic","limit":-1},"encodings":{"dimensions":[{"dragId":"gw_5Dux","fid":"state_node","name":"state_node","basename":"state_node","semanticType":"nominal","analyticType":"dimension"},{"dragId":"gw__GB7","fid":"interval_end","name":"interval_end","basename":"interval_end","semanticType":"ordinal","analyticType":"dimension"},{"dragId":"gw_Xb3-","fid":"node_grp","name":"node_grp","basename":"node_grp","semanticType":"nominal","analyticType":"dimension"},{"dragId":"gw_IGtN","fid":"gadm_adm0","name":"gadm_adm0","basename":"gadm_adm0","semanticType":"quantitative","analyticType":"dimension"},{"dragId":"gw_848c","fid":"meaning","name":"meaning","basename":"meaning","semanticType":"nominal","analyticType":"dimension"},{"dragId":"gw_mea_key_fid","fid":"gw_mea_key_fid","name":"Measure names","analyticType":"dimension","semanticType":"nominal"}],"measures":[{"dragId":"gw_y99d","fid":"area__ha","name":"area__ha","basename":"area__ha","analyticType":"measure","semanticType":"quantitative","aggName":"sum"},{"dragId":"gw_Sjqt","fid":"gross_emissions__all_C_pools__CO2_only__MgCO2","name":"gross_emissions__all_C_pools__CO2_only__MgCO2","basename":"gross_emissions__all_C_pools__CO2_only__MgCO2","analyticType":"measure","semanticType":"quantitative","aggName":"sum"},{"dragId":"gw_5GlR","fid":"gross_emissions__all_C_pools__CO2_only__MgCO2__C_per_ha","name":"gross_emissions__all_C_pools__CO2_only__MgCO2__C_per_ha","basename":"gross_emissions__all_C_pools__CO2_only__MgCO2__C_per_ha","analyticType":"measure","semanticType":"quantitative","aggName":"sum"},{"dragId":"gw_axS0","fid":"gross_emissions__all_C_pools__all_gases__MgCO2e","name":"gross_emissions__all_C_pools__all_gases__MgCO2e","basename":"gross_emissions__all_C_pools__all_gases__MgCO2e","analyticType":"measure","semanticType":"quantitative","aggName":"sum"},{"dragId":"gw_GZnF","fid":"gross_emissions__all_C_pools__all_gases__MgCO2e__C_per_ha","name":"gross_emissions__all_C_pools__all_gases__MgCO2e__C_per_ha","basename":"gross_emissions__all_C_pools__all_gases__MgCO2e__C_per_ha","analyticType":"measure","semanticType":"quantitative","aggName":"sum"},{"dragId":"gw_cuXy","fid":"gross_removals__all_C_pools__MgCO2","name":"gross_removals__all_C_pools__MgCO2","basename":"gross_removals__all_C_pools__MgCO2","analyticType":"measure","semanticType":"quantitative","aggName":"sum"},{"dragId":"gw_t1MS","fid":"gross_removals__all_C_pools__MgCO2__C_per_ha","name":"gross_removals__all_C_pools__MgCO2__C_per_ha","basename":"gross_removals__all_C_pools__MgCO2__C_per_ha","analyticType":"measure","semanticType":"quantitative","aggName":"sum"},{"dragId":"gw_12Yi","fid":"net_flux__all_C_pools__CO2_only__MgCO2","name":"net_flux__all_C_pools__CO2_only__MgCO2","basename":"net_flux__all_C_pools__CO2_only__MgCO2","analyticType":"measure","semanticType":"quantitative","aggName":"sum"},{"dragId":"gw_e6y_","fid":"net_flux__all_C_pools__CO2_only__MgCO2__C_per_ha","name":"net_flux__all_C_pools__CO2_only__MgCO2__C_per_ha","basename":"net_flux__all_C_pools__CO2_only__MgCO2__C_per_ha","analyticType":"measure","semanticType":"quantitative","aggName":"sum"},{"dragId":"gw_count_fid","fid":"gw_count_fid","name":"Row count","analyticType":"measure","semanticType":"quantitative","aggName":"sum","computed":true,"expression":{"op":"one","params":[],"as":"gw_count_fid"}},{"dragId":"gw_mea_val_fid","fid":"gw_mea_val_fid","name":"Measure values","analyticType":"measure","semanticType":"quantitative","aggName":"sum"}],"rows":[{"dragId":"gw_fCJt","fid":"area__ha","name":"area__ha","basename":"area__ha","analyticType":"measure","semanticType":"quantitative","aggName":"sum"}],"columns":[{"dragId":"gw_uU2i","fid":"interval_end","name":"interval_end","basename":"interval_end","semanticType":"ordinal","analyticType":"dimension"}],"color":[{"dragId":"gw_DITZ","fid":"meaning","name":"meaning","basename":"meaning","semanticType":"nominal","analyticType":"dimension"}],"opacity":[],"size":[],"shape":[],"radius":[],"theta":[],"longitude":[],"latitude":[],"geoId":[],"details":[],"filters":[{"dragId":"gw_GO_f","fid":"gadm_adm0","name":"gadm_adm0","basename":"gadm_adm0","semanticType":"quantitative","analyticType":"dimension","rule":{"type":"one of","value":[180]}},{"dragId":"gw_c6Cy","fid":"meaning","name":"meaning","basename":"meaning","semanticType":"nominal","analyticType":"dimension","rule":{"type":"one of","value":["<100 year old natural forest not disturbed in last interval, with fire",">100 year old natural forest not disturbed in last interval, with fire","Natural forest converted to short vegetation with disturbance that emits all non-soil C pools, without fire","Forest partially disturbed in the last interval without signif. height increase after, without fire"]}}],"text":[]},"layout":{"showActions":false,"showTableSummary":false,"stack":"stack","interactiveScale":false,"zeroScale":true,"size":{"mode":"auto","width":320,"height":200},"format":{},"geoKey":"name","resolve":{"x":false,"y":false,"color":false,"opacity":false,"shape":false,"size":false}},"visId":"gw_iAyO","name":"Emissions area"},{"config":{"defaultAggregated":true,"geoms":["line"],"coordSystem":"generic","limit":-1},"encodings":{"dimensions":[{"dragId":"gw_5Dux","fid":"state_node","name":"state_node","basename":"state_node","semanticType":"nominal","analyticType":"dimension"},{"dragId":"gw__GB7","fid":"interval_end","name":"interval_end","basename":"interval_end","semanticType":"ordinal","analyticType":"dimension"},{"dragId":"gw_Xb3-","fid":"node_grp","name":"node_grp","basename":"node_grp","semanticType":"nominal","analyticType":"dimension"},{"dragId":"gw_IGtN","fid":"gadm_adm0","name":"gadm_adm0","basename":"gadm_adm0","semanticType":"quantitative","analyticType":"dimension"},{"dragId":"gw_848c","fid":"meaning","name":"meaning","basename":"meaning","semanticType":"nominal","analyticType":"dimension"},{"dragId":"gw_mea_key_fid","fid":"gw_mea_key_fid","name":"Measure names","analyticType":"dimension","semanticType":"nominal"}],"measures":[{"dragId":"gw_y99d","fid":"area__ha","name":"area__ha","basename":"area__ha","analyticType":"measure","semanticType":"quantitative","aggName":"sum"},{"dragId":"gw_Sjqt","fid":"gross_emissions__all_C_pools__CO2_only__MgCO2","name":"gross_emissions__all_C_pools__CO2_only__MgCO2","basename":"gross_emissions__all_C_pools__CO2_only__MgCO2","analyticType":"measure","semanticType":"quantitative","aggName":"sum"},{"dragId":"gw_5GlR","fid":"gross_emissions__all_C_pools__CO2_only__MgCO2__C_per_ha","name":"gross_emissions__all_C_pools__CO2_only__MgCO2__C_per_ha","basename":"gross_emissions__all_C_pools__CO2_only__MgCO2__C_per_ha","analyticType":"measure","semanticType":"quantitative","aggName":"sum"},{"dragId":"gw_axS0","fid":"gross_emissions__all_C_pools__all_gases__MgCO2e","name":"gross_emissions__all_C_pools__all_gases__MgCO2e","basename":"gross_emissions__all_C_pools__all_gases__MgCO2e","analyticType":"measure","semanticType":"quantitative","aggName":"sum"},{"dragId":"gw_GZnF","fid":"gross_emissions__all_C_pools__all_gases__MgCO2e__C_per_ha","name":"gross_emissions__all_C_pools__all_gases__MgCO2e__C_per_ha","basename":"gross_emissions__all_C_pools__all_gases__MgCO2e__C_per_ha","analyticType":"measure","semanticType":"quantitative","aggName":"sum"},{"dragId":"gw_cuXy","fid":"gross_removals__all_C_pools__MgCO2","name":"gross_removals__all_C_pools__MgCO2","basename":"gross_removals__all_C_pools__MgCO2","analyticType":"measure","semanticType":"quantitative","aggName":"sum"},{"dragId":"gw_t1MS","fid":"gross_removals__all_C_pools__MgCO2__C_per_ha","name":"gross_removals__all_C_pools__MgCO2__C_per_ha","basename":"gross_removals__all_C_pools__MgCO2__C_per_ha","analyticType":"measure","semanticType":"quantitative","aggName":"sum"},{"dragId":"gw_12Yi","fid":"net_flux__all_C_pools__CO2_only__MgCO2","name":"net_flux__all_C_pools__CO2_only__MgCO2","basename":"net_flux__all_C_pools__CO2_only__MgCO2","analyticType":"measure","semanticType":"quantitative","aggName":"sum"},{"dragId":"gw_e6y_","fid":"net_flux__all_C_pools__CO2_only__MgCO2__C_per_ha","name":"net_flux__all_C_pools__CO2_only__MgCO2__C_per_ha","basename":"net_flux__all_C_pools__CO2_only__MgCO2__C_per_ha","analyticType":"measure","semanticType":"quantitative","aggName":"sum"},{"dragId":"gw_count_fid","fid":"gw_count_fid","name":"Row count","analyticType":"measure","semanticType":"quantitative","aggName":"sum","computed":true,"expression":{"op":"one","params":[],"as":"gw_count_fid"}},{"dragId":"gw_mea_val_fid","fid":"gw_mea_val_fid","name":"Measure values","analyticType":"measure","semanticType":"quantitative","aggName":"sum"}],"rows":[{"dragId":"gw_rnBz","fid":"gross_emissions__all_C_pools__CO2_only__MgCO2","name":"gross_emissions__all_C_pools__CO2_only__MgCO2","basename":"gross_emissions__all_C_pools__CO2_only__MgCO2","analyticType":"measure","semanticType":"quantitative","aggName":"sum"},{"dragId":"gw_7xrz","fid":"gross_emissions__all_C_pools__CO2_only__MgCO2__C_per_ha","name":"gross_emissions__all_C_pools__CO2_only__MgCO2__C_per_ha","basename":"gross_emissions__all_C_pools__CO2_only__MgCO2__C_per_ha","analyticType":"measure","semanticType":"quantitative","aggName":"sum"},{"dragId":"gw_fCJt","fid":"area__ha","name":"area__ha","basename":"area__ha","analyticType":"measure","semanticType":"quantitative","aggName":"sum"}],"columns":[{"dragId":"gw_uU2i","fid":"interval_end","name":"interval_end","basename":"interval_end","semanticType":"ordinal","analyticType":"dimension"}],"color":[{"dragId":"gw_DITZ","fid":"meaning","name":"meaning","basename":"meaning","semanticType":"nominal","analyticType":"dimension"}],"opacity":[],"size":[],"shape":[],"radius":[],"theta":[],"longitude":[],"latitude":[],"geoId":[],"details":[],"filters":[{"dragId":"gw_GO_f","fid":"gadm_adm0","name":"gadm_adm0","basename":"gadm_adm0","semanticType":"quantitative","analyticType":"dimension","rule":{"type":"one of","value":[180]}},{"dragId":"gw_c6Cy","fid":"meaning","name":"meaning","basename":"meaning","semanticType":"nominal","analyticType":"dimension","rule":{"type":"one of","value":["<100 year old natural forest not disturbed in last interval, with fire",">100 year old natural forest not disturbed in last interval, with fire","Natural forest converted to short vegetation with disturbance that emits all non-soil C pools, without fire","Forest partially disturbed in the last interval without signif. height increase after, without fire"]}}],"text":[]},"layout":{"showActions":false,"showTableSummary":false,"stack":"stack","interactiveScale":false,"zeroScale":true,"size":{"mode":"auto","width":320,"height":200},"format":{},"geoKey":"name","resolve":{"x":false,"y":false,"color":false,"opacity":false,"shape":false,"size":false}},"visId":"gw_MCCS","name":"All three panels"}],"chart_map":{},"workflow_list":[{"workflow":[{"type":"filter","filters":[{"fid":"gadm_adm0","rule":{"type":"one of","value":[180]}},{"fid":"meaning","rule":{"type":"one of","value":["<100 year old natural forest not disturbed in last interval, with fire",">100 year old natural forest not disturbed in last interval, with fire","Natural forest converted to short vegetation with disturbance that emits all non-soil C pools, without fire","Forest partially disturbed in the last interval without signif. height increase after, without fire"]}}]},{"type":"view","query":[{"op":"aggregate","groupBy":["interval_end","meaning"],"measures":[{"field":"gross_emissions__all_C_pools__CO2_only__MgCO2","agg":"sum","asFieldKey":"gross_emissions__all_C_pools__CO2_only__MgCO2_sum"}]}]}]},{"workflow":[{"type":"filter","filters":[{"fid":"gadm_adm0","rule":{"type":"one of","value":[180]}},{"fid":"meaning","rule":{"type":"one of","value":["<100 year old natural forest not disturbed in last interval, with fire",">100 year old natural forest not disturbed in last interval, with fire","Natural forest converted to short vegetation with disturbance that emits all non-soil C pools, without fire","Forest partially disturbed in the last interval without signif. height increase after, without fire"]}}]},{"type":"view","query":[{"op":"aggregate","groupBy":["interval_end","meaning"],"measures":[{"field":"gross_emissions__all_C_pools__CO2_only__MgCO2__C_per_ha","agg":"sum","asFieldKey":"gross_emissions__all_C_pools__CO2_only__MgCO2__C_per_ha_sum"}]}]}]},{"workflow":[{"type":"filter","filters":[{"fid":"gadm_adm0","rule":{"type":"one of","value":[180]}},{"fid":"meaning","rule":{"type":"one of","value":["<100 year old natural forest not disturbed in last interval, with fire",">100 year old natural forest not disturbed in last interval, with fire","Natural forest converted to short vegetation with disturbance that emits all non-soil C pools, without fire","Forest partially disturbed in the last interval without signif. height increase after, without fire"]}}]},{"type":"view","query":[{"op":"aggregate","groupBy":["interval_end","meaning"],"measures":[{"field":"area__ha","agg":"sum","asFieldKey":"area__ha_sum"}]}]}]},{"workflow":[{"type":"filter","filters":[{"fid":"gadm_adm0","rule":{"type":"one of","value":[180]}},{"fid":"meaning","rule":{"type":"one of","value":["<100 year old natural forest not disturbed in last interval, with fire",">100 year old natural forest not disturbed in last interval, with fire","Natural forest converted to short vegetation with disturbance that emits all non-soil C pools, without fire","Forest partially disturbed in the last interval without signif. height increase after, without fire"]}}]},{"type":"view","query":[{"op":"aggregate","groupBy":["interval_end","meaning"],"measures":[{"field":"gross_emissions__all_C_pools__CO2_only__MgCO2","agg":"sum","asFieldKey":"gross_emissions__all_C_pools__CO2_only__MgCO2_sum"},{"field":"gross_emissions__all_C_pools__CO2_only__MgCO2__C_per_ha","agg":"sum","asFieldKey":"gross_emissions__all_C_pools__CO2_only__MgCO2__C_per_ha_sum"},{"field":"area__ha","agg":"sum","asFieldKey":"area__ha_sum"}]}]}]}],"timezoneOffsetSeconds":-14400,"version":"0.3.17"}"""
pyg.walk(combined_df_wide, spec=vis_spec)